In [1]:
import zipfile
import os 

zip_path = "AudioData.zip"

extract_to = "Audio1"

os.makedirs(extract_to , exist_ok = True)

with zipfile.ZipFile("AudioData.zip" , "r") as zip_ref :
    zip_ref.extractall(extract_to)
    

In [2]:
pip install librosa pydub

Note: you may need to restart the kernel to use updated packages.


In [3]:
import librosa
import librosa.display
import IPython.display as ipd
import os
from IPython.display import Audio

In [6]:
Audio(filename="Audio1/AudioData/A1_1.wav")

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [9]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
from scipy.signal import medfilt


folder_path = "Audio1/AudioData"  
all_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.wav')])

features = []
labels = []

def preprocess_and_extract(file_path):
    y, sr = librosa.load(file_path, sr=None)

    y = medfilt(y, kernel_size=3)

    
    intervals = librosa.effects.split(y, top_db=30)
    y = np.concatenate([y[start:end] for start, end in intervals])

    y = librosa.util.normalize(y)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    zero_crossings = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)

    feature_vector = [
        *np.mean(mfcc, axis=1),
        np.mean(spectral_centroid),
        np.mean(spectral_rolloff),
        np.mean(zero_crossings),
        np.mean(rms)
    ]
    
    return np.array(feature_vector)

for i, filename in enumerate(all_files):
    file_path = os.path.join(folder_path, filename)

    label = 0 if i < 190 else 1

    try:
        feats = preprocess_and_extract(file_path)
        features.append(feats)
        labels.append(label)
    except Exception as e:
        print(f"Error processing {filename}: {e}")

columns = [f"MFCC_{i+1}" for i in range(13)] + [
    "Spectral_Centroid", "Spectral_Rolloff", "Zero_Crossing_Rate", "RMS"
]
df = pd.DataFrame(features, columns=columns)
df["Label"] = labels

df.to_csv("preprocessed_audio_features1.csv",index=False)

In [10]:
df = pd.read_csv("preprocessed_audio_features1.csv")

df

,MFCC_1,MFCC_2,MFCC_3,MFCC_4,MFCC_5,MFCC_6,MFCC_7,MFCC_8,MFCC_9,MFCC_10,MFCC_11,MFCC_12,MFCC_13,Spectral_Centroid,Spectral_Rolloff,Zero_Crossing_Rate,RMS,Label
0,-187.809250,117.227501,-31.110760,15.556813,-7.885459,2.843250,-9.991956,-21.498390,0.770710,-17.766630,-10.855116,-3.887883,-10.148057,3179.515284,5778.299877,0.066818,0.144618,0
1,-188.325394,107.297447,-30.705948,12.014100,-8.016599,8.633282,-7.907493,-22.088537,3.992622,-17.290718,-11.506030,-5.717010,-10.652798,3544.943704,6292.618313,0.078872,0.136885,0
2,-210.070404,108.465698,-33.074478,16.595703,-10.307270,0.398625,-11.207914,-22.037777,3.446383,-16.117588,-11.892857,-3.206302,-10.391039,3275.955929,6062.904680,0.066980,0.126447,0
3,-191.454346,114.692177,-30.946205,14.612432,-6.709668,5.452821,-8.977777,-20.556648,1.169109,-17.638140,-11.115117,-7.023687,-11.559093,3233.029510,5822.119652,0.067483,0.138934,0
4,-205.284378,120.310707,-31.185942,13.799322,-1.770789,7.155611,-9.524747,-17.709455,1.966191,-16.525158,-8.841122,-6.653839,-13.629004,3042.808361,5509.687653,0.058151,0.119066,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,-235.970673,92.480553,-17.970011,36.248577,7.397984,-1.366915,-5.079729,-4.358680,2.995737,-13.785440,-0.208047,-3.909730,-10.535037,3648.659581,6665.867448,0.067040,0.100079,1
396,-228.613861,89.041344,-20.342937,37.032528,7.664256,1.912295,-3.883863,-6.076528,4.707572,-11.078963,-0.212174,-2.637521,-8.992634,3779.143344,6861.442132,0.073427,0.101179,1
397,-243.517639,95.312218,-16.581459,40.670311,15.478923,2.264346,-6.612248,-4.203165,3.711258,-12.687075,-0.346347,-5.403016,-11.292534,3528.143882,6421.854369,0.065456,0.092043,1
398,-227.889404,99.602150,-17.395329,38.284962,14.276464,1.938467,-6.511340,-4.297634,1.381614,-10.693131,3.438414,-5.950038,-10.636674,3448.686427,6288.920255,0.061266,0.108516,1


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   MFCC_1              400 non-null    float64
 1   MFCC_2              400 non-null    float64
 2   MFCC_3              400 non-null    float64
 3   MFCC_4              400 non-null    float64
 4   MFCC_5              400 non-null    float64
 5   MFCC_6              400 non-null    float64
 6   MFCC_7              400 non-null    float64
 7   MFCC_8              400 non-null    float64
 8   MFCC_9              400 non-null    float64
 9   MFCC_10             400 non-null    float64
 10  MFCC_11             400 non-null    float64
 11  MFCC_12             400 non-null    float64
 12  MFCC_13             400 non-null    float64
 13  Spectral_Centroid   400 non-null    float64
 14  Spectral_Rolloff    400 non-null    float64
 15  Zero_Crossing_Rate  400 non-null    float64
 16  RMS     

In [12]:
df.describe()

,MFCC_1,MFCC_2,MFCC_3,MFCC_4,MFCC_5,MFCC_6,MFCC_7,MFCC_8,MFCC_9,MFCC_10,MFCC_11,MFCC_12,MFCC_13,Spectral_Centroid,Spectral_Rolloff,Zero_Crossing_Rate,RMS,Label
count,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000
mean,-259.785668,140.316405,-15.295260,5.728152,3.511850,-2.897901,-15.015573,-11.346217,-7.435469,-12.796526,-8.218167,-7.782337,-8.782769,2359.724400,3994.096736,0.049233,0.116311,0.525
std,46.370038,24.724280,23.334195,17.358977,12.608287,11.388969,9.280171,7.707574,6.832294,5.435688,6.748178,5.566851,5.142823,597.173745,1069.215734,0.014423,0.052569,0.500
min,-361.561035,84.376282,-76.867737,-48.840527,-30.801674,-43.195805,-34.780884,-36.211220,-23.282700,-25.613447,-24.998173,-19.621284,-23.180586,1076.001366,1593.091619,0.023760,0.039793,0.000
25%,-295.182381,120.523771,-32.820293,-4.566481,-4.847082,-9.123543,-22.274520,-15.310081,-12.736340,-17.296555,-13.646011,-11.810560,-12.711378,1970.023865,3318.654336,0.037777,0.078074,0.000
50%,-262.296005,139.550659,-14.656782,8.306437,2.451574,-1.104372,-15.247597,-9.768071,-7.619180,-12.769848,-6.926264,-8.080667,-8.835824,2319.494744,3964.159102,0.047734,0.102862,1.000
75%,-228.672352,159.493256,-1.628636,16.532336,12.961899,5.182067,-8.543664,-6.278350,-1.485344,-8.545326,-3.174964,-3.896649,-4.656793,2740.368638,4654.091550,0.058782,0.143079,1.000
max,-75.975716,200.338165,36.941036,47.650021,32.760479,22.705479,13.234770,6.147036,8.418449,1.755651,3.759895,10.225003,1.803776,3851.282494,6900.088249,0.099408,0.368804,1.000


In [13]:
X = df.drop(columns = "Label")
y = df["Label"]
X.head()

,MFCC_1,MFCC_2,MFCC_3,MFCC_4,MFCC_5,MFCC_6,MFCC_7,MFCC_8,MFCC_9,MFCC_10,MFCC_11,MFCC_12,MFCC_13,Spectral_Centroid,Spectral_Rolloff,Zero_Crossing_Rate,RMS
0,-187.809250,117.227501,-31.110760,15.556813,-7.885459,2.843250,-9.991956,-21.498390,0.770710,-17.766630,-10.855116,-3.887883,-10.148057,3179.515284,5778.299877,0.066818,0.144618
1,-188.325394,107.297447,-30.705948,12.014100,-8.016599,8.633282,-7.907493,-22.088537,3.992622,-17.290718,-11.506030,-5.717010,-10.652798,3544.943704,6292.618313,0.078872,0.136885
2,-210.070404,108.465698,-33.074478,16.595703,-10.307270,0.398625,-11.207914,-22.037777,3.446383,-16.117588,-11.892857,-3.206302,-10.391039,3275.955929,6062.904680,0.066980,0.126447
3,-191.454346,114.692177,-30.946205,14.612432,-6.709668,5.452821,-8.977777,-20.556648,1.169109,-17.638140,-11.115117,-7.023687,-11.559093,3233.029510,5822.119652,0.067483,0.138934
4,-205.284378,120.310707,-31.185942,13.799322,-1.770789,7.155611,-9.524747,-17.709455,1.966191,-16.525158,-8.841122,-6.653839,-13.629004,3042.808361,5509.687653,0.058151,0.119066


In [14]:
from sklearn.model_selection import train_test_split

X_train , X_test ,  y_train , y_test = train_test_split(X , y , test_size = 0.3 , random_state = 42)

In [ ]:
import numpy as np

# Training function
def fit_naive_bayes(X, y):
    n_samples, n_features = X.shape
    classes = np.unique(y)
    n_classes = len(classes)

    Avg = np.zeros((n_classes, n_features))
    Variance = np.zeros((n_classes, n_features))
    priors = np.zeros(n_classes)

    for i, label in enumerate(classes):
        X_c = X[y == label]
        Avg[i, :] = X_c.mean(axis=0)
        Variance[i, :] = X_c.var(axis=0)
        priors[i] = X_c.shape[0] / float(n_samples)

    return classes, Avg, Variance, priors

# Likelihood function
def like_hood(point, Avg, Variance):
    return (1 / np.sqrt(2 * np.pi * Variance)) * np.exp(-((point - Avg) ** 2) / (2 * Variance))

# Posterior calculation
def compute_posterior(X_point, classes, Avg, Variance, priors):
    posteriors = []

    for index, label in enumerate(classes):
        likelihoods = like_hood(X_point, Avg[index], Variance[index])
        posterior = priors[index] * np.prod(likelihoods)
        posteriors.append((index, posterior))

    best_index = max(posteriors, key=lambda x: x[1])[0]
    return classes[best_index]

# Prediction function
def predict_naive_bayes(X_test, classes, Avg, Variance, priors):
    predictions = []
    for i in range(len(X_test)):
        point = X_test[i] if isinstance(X_test, np.ndarray) else X_test.iloc[i].values     # This line handles two types of input: NumPy arrays and Pandas DataFrames.

                                                                                         # If it's a NumPy array, directly access the row.

                                                                                         # If it's a DataFrame, use .iloc[i].values to get the row as a NumPy array.
        pred = compute_posterior(point, classes, Avg, Variance, priors)
        predictions.append(pred)
    return predictions

# Fit the model
classes, Avg, Variance, priors = fit_naive_bayes(X_train, y_train)

# Predict
y_pred = predict_naive_bayes(X_test, classes, Avg, Variance, priors)


In [16]:
from sklearn import metrics
metrics.confusion_matrix(y_test,y_pred)

array([[49, 12],
       [ 5, 54]], dtype=int64)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Naïve Bayes From Scratch Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))


Naïve Bayes From Scratch Performance:
Accuracy: 0.8583333333333333
Precision: 0.8181818181818182
Recall: 0.9152542372881356
F1 Score: 0.864


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
from collections import Counter

# Bagging function
def bagging_ensemble(X_train, y_train, X_test, base_model='naive_bayes', n_estimators=10):
    all_preds = []

    for i in range(n_estimators):
        X_sample, y_sample = resample(X_train, y_train, replace=True, random_state=i)

        if base_model == 'naive_bayes':
            classes, Avg, Variance, priors = fit_naive_bayes(X_sample, y_sample)
            preds = predict_naive_bayes(X_test, classes, Avg, Variance, priors)

        elif base_model == 'logistic_regression':
            model = LogisticRegression()
            model.fit(X_sample, y_sample)
            preds = model.predict(X_test)

        all_preds.append(preds)

    # Majority Voting
    final_preds = []
    for i in range(len(X_test)):
        votes = [all_preds[j][i] for j in range(n_estimators)]
        final_preds.append(Counter(votes).most_common(1)[0][0])
    
    return final_preds

# Bagging with Naive Bayes
y_pred_bag_nb = bagging_ensemble(X_train, y_train, X_test, base_model='naive_bayes', n_estimators=10)

# Bagging with Logistic Regression
y_pred_bag_lr = bagging_ensemble(X_train, y_train, X_test, base_model='logistic_regression', n_estimators=10)

# Evaluate
print("\nBagging Naïve Bayes Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_bag_nb))
print("Precision:", precision_score(y_test, y_pred_bag_nb))
print("Recall:", recall_score(y_test, y_pred_bag_nb))
print("F1 Score:", f1_score(y_test, y_pred_bag_nb))

print("\nBagging Logistic Regression Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_bag_lr))
print("Precision:", precision_score(y_test, y_pred_bag_lr))
print("Recall:", recall_score(y_test, y_pred_bag_lr))
print("F1 Score:", f1_score(y_test, y_pred_bag_lr))



Bagging Naïve Bayes Performance:
Accuracy: 0.8833333333333333
Precision: 0.8461538461538461
Recall: 0.9322033898305084
F1 Score: 0.8870967741935484

Bagging Logistic Regression Performance:
Accuracy: 0.9333333333333333
Precision: 0.8923076923076924
Recall: 0.9830508474576272
F1 Score: 0.9354838709677419


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stab